We'll start by scraping the web for nutritional info on various products.

In [32]:
!pip install requests beautifulsoup4
!pip install lxml
!pip install selenium
!pip install pandas


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\andre\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\andre\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\andre\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
    --------------------------------------- 0.2/9.9 MB 3.5 MB/s eta 0:00:03
   - -------------------------------------- 0.3/9.9 MB 2.4 MB/s eta 0:00:04
   - -------------------------------------- 0.3/9.9 MB 2.0 MB/s eta 0:00:05
   - -------------------------------------- 0.5/9.9 MB 2.3 MB/s eta 0:00:05
   ---- ----------------------------------- 1.1/9.9 MB 4.2 MB/s eta 0:00:03
   ------- -------------------------------- 1.9/9.9 MB 6.3 MB/s eta 0:00:02
   ----------- ---------------------------- 3.0/9.9 MB 8.6 MB/s eta 0:00:01
   ----------------- ---------------------- 4.4/9.9 MB 11.1 MB/s eta 0:00:01
   ----------------------- ---------------- 5.8/9.9 MB 13.3 MB/s eta 0:00:01
   ------------------------------ --------- 7.5/9.9 MB 15.4 MB/s eta 0:00:01
   ------------------------------------ --- 8.9/9.9 MB 16.8 MB/s eta 0:00:01
   ---------------------------------------  9.9/9.9 MB 17.6 MB/s eta 0:00:01
   -----------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\andre\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [33]:
import requests
from bs4 import BeautifulSoup
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd

In [2]:
# get a list of all the nutrients we care about

nutrients = ["Protein", "Sodium", "Saturated Fat", "Dietary Fiber",
            "Iron", "Vitamin B12", "Cholesterol"]

In [3]:
# Beyond Meat

# function that gets all pages with nutrition info
def getBeyondUrls():
    response = requests.get("https://www.beyondmeat.com/sitemap.xml")
    soup = BeautifulSoup(response.text, "xml")

    # get all the URLs
    locs = soup.find_all("loc")
    all_urls = [loc.text for loc in locs]

    # get product URLs
    product_urls = [url for url in all_urls if url.startswith("https://www.beyondmeat.com/en-US/products/")]

    return product_urls


# function that scrapes a url for nutrition information
def beyondMeatScraper(url):

    response = requests.get(url)
    
    # if unable to fetch web page
    if response.status_code != 200:
        return None
    
    # if successful
    soup = BeautifulSoup(response.text, "html.parser")

    # create a dictionary to store information about nutrients
    nutrientDict = {}

    # investigate calories separately
    match = re.search(r"Calories[:\s]*(\d+)", soup.get_text())
    if match:
        nutrientDict["Calories"] = match.group(1)
    else:
        return None
    
    # find a tag with the td (table data) label that has a given nutrient
    for nut in nutrients:
        # finds a tag <td> such that inside is the exact string             
        nut_label = soup.find("td", string=nut)

        # deal with specific saturated fat typo
        if nut == "Saturated Fat" and nut_label == None:
            nut_label = soup.find("td", string="Satured Fat")

        # gets the next <td> in the same row unless the nutrient isn't in the table
        try:
            nut_value = nut_label.find_next_sibling("td")
            nutrientDict.update({nut:nut_value.text})
        except:
            nutrientDict.update({nut:"N/A"})

    # make sure we know what url this corresponds to
    nutrientDict["URL"] = url

    # add brand
    nutrientDict["Brand"] = "Beyond"

    # fill in product 
    productName = url.split("/products/")[1].replace("/", " ").replace("-", " ").title()
    nutrientDict["Product"] = productName

    return nutrientDict



In [4]:
# get all the results

beyondResults = []
for url in getBeyondUrls():
    d = beyondMeatScraper(url)
    if d is not None:
        beyondResults.append(d)

In [5]:
# get rid of repeats and products that don't seem to exist anymore

# compile extraneous urls

extraneousURLs = [
    "https://www.beyondmeat.com/en-US/products/beyond-meatballs",
    "https://www.beyondmeat.com/en-US/products/beyond-sun-sausage",
    "https://www.beyondmeat.com/en-US/products/beyond-steak/beyond-steak",
    "https://www.beyondmeat.com/en-US/products/beyond-stack-burger",
    "https://www.beyondmeat.com/en-US/products/beyond-chicken-nuggets",
    "https://www.beyondmeat.com/en-US/products/beyond-breakfast-sausage",
    "https://www.beyondmeat.com/en-US/products/the-beyond-burger",
    "https://www.beyondmeat.com/en-US/products/beyond-beef",
    "https://www.beyondmeat.com/en-US/products/beyond-sausage",
    "https://www.beyondmeat.com/en-US/products/beyond-sausage/hot-italian-style",
    "https://www.beyondmeat.com/en-US/products/beyond-sausage/brat",
    "https://www.beyondmeat.com/en-US/products/beyond-chicken",
    "https://www.beyondmeat.com/en-US/products/beyond-chicken/chicken-spicy-buffalo",
    "https://www.beyondmeat.com/en-US/products/beyond-chicken/lemon-herb"
]

# get rid of anything in beyondResults that has url equal to something in extraneousURLs

for nutritionalInfo in beyondResults:
    for url in extraneousURLs:
        if nutritionalInfo['URL'] == url:
            beyondResults.remove(nutritionalInfo)


In [6]:
beyondResults[6]['Product'] = "Beyond Stack Burger"
beyondResults[7]['Product'] = "Beyond Chicken Nuggets"
beyondResults[11]['Product'] = "Beyond Burger"
beyondResults[17]['Product'] = "Beyond Chicken Pieces"

beyondResults

[{'Calories': '290',
  'Protein': '19 g',
  'Sodium': '500 mg',
  'Saturated Fat': '7 g',
  'Dietary Fiber': '3 g',
  'Iron': '4.9mg',
  'Vitamin B12': 'N/A',
  'Cholesterol': '0 mg',
  'URL': 'https://www.beyondmeat.com/en-US/products/beyond-meatballs/italian-style',
  'Brand': 'Beyond',
  'Product': 'Beyond Meatballs Italian Style'},
 {'Calories': '140',
  'Protein': '12g',
  'Sodium': '320mg',
  'Saturated Fat': '1g',
  'Dietary Fiber': '1g',
  'Iron': '2.8mg',
  'Vitamin B12': 'N/A',
  'Cholesterol': '0g',
  'URL': 'https://www.beyondmeat.com/en-US/products/beyond-sun-sausage/pesto',
  'Brand': 'Beyond',
  'Product': 'Beyond Sun Sausage Pesto'},
 {'Calories': '140',
  'Protein': '12g',
  'Sodium': '320mg',
  'Saturated Fat': '1g',
  'Dietary Fiber': '1g',
  'Iron': '2.6mg',
  'Vitamin B12': 'N/A',
  'Cholesterol': '0g',
  'URL': 'https://www.beyondmeat.com/en-US/products/beyond-sun-sausage/pineapple-jalapeno',
  'Brand': 'Beyond',
  'Product': 'Beyond Sun Sausage Pineapple Jalapeno

In [7]:
# Impossible Foods

# we'll input the data manually here. the nutrition info is stored in pictures.

impossibleResults = [
    {
        'Calories': 230,
        'Protein': '19g',
        'Sodium': '370mg',
        'Saturated Fat': '6g',
        'Dietary Fiber': '5g',
        'Iron': '4.2mg',
        'Vitamin B12': '130%',
        'Cholesterol': '0mg',
        'URL': 'https://faq.impossiblefoods.com/hc/en-us/articles/360018939274-What-are-the-nutrition-facts-for-Impossible-Beef-Meat-From-Plants',
        'Brand': 'Impossible Foods',
        'Product': 'Impossible Beef'
    },
    {
        'Calories': 170,
        'Protein': '21g',
        'Sodium': '420mg',
        'Saturated Fat': '0.5g',
        'Dietary Fiber': '3g',
        'Iron': '3.1mg',
        'Vitamin B12': '80%',
        'Cholesterol': '0mg',
        'URL': 'https://faq.impossiblefoods.com/hc/en-us/articles/30387467337879-What-are-the-nutrition-facts-for-Impossible-Steak-Bites-Meat-From-Plants',
        'Brand': 'Impossible Foods',
        'Product': 'Impossible Steak Bites'
    },
    {
        'Calories': 120,
        'Protein': '12g',
        'Sodium': '430mg',
        'Saturated Fat': '2.5g',
        'Dietary Fiber': '0g',
        'Iron': '1mg',
        'Vitamin B12': '50%',
        'Cholesterol': '0mg',
        'URL': 'https://faq.impossiblefoods.com/hc/en-us/articles/19576943205015-What-are-the-nutrition-facts-for-Impossible-Beef-Hot-Dogs-Meat-From-Plants',
        'Brand': 'Impossible Foods',
        'Product': 'Impossible Beef Hot Dogs'
    },
    {
        'Calories': 190,
        'Protein': '9g',
        'Sodium': '470mg',
        'Saturated Fat': '2g',
        'Dietary Fiber': '1g',
        'Iron': '0.9mg',
        'Vitamin B12': '40%',
        'Cholesterol': '0mg',
        'URL': 'https://faq.impossiblefoods.com/hc/en-us/articles/26714794517399-What-are-the-nutrition-facts-for-Impossible-Corn-Dogs-Meat-From-Plants',
        'Brand': 'Impossible Foods',
        'Product': 'Impossible Corn Dogs'
    },
    {
        'Calories': 130,
        'Protein': '7g',
        'Sodium': '380mg',
        'Saturated Fat': '4g',
        'Dietary Fiber': '1g',
        'Iron': '1.4mg',
        'Vitamin B12': '50%',
        'Cholesterol': '0mg',
        'URL': 'https://faq.impossiblefoods.com/hc/en-us/articles/1500011067082-What-are-the-nutrition-facts-for-Impossible-Sausage-Meat-From-Plants',
        'Brand': 'Impossible Foods',
        'Product': 'Impossible Savory Sausage'
    },
    {
        'Calories': 130,
        'Protein': '7g',
        'Sodium': '370mg',
        'Saturated Fat': '4g',
        'Dietary Fiber': '1g',
        'Iron': '1.5mg',
        'Vitamin B12': '50%',
        'Cholesterol': '0mg',
        'URL': 'https://faq.impossiblefoods.com/hc/en-us/articles/1500011067082-What-are-the-nutrition-facts-for-Impossible-Sausage-Meat-From-Plants',
        'Brand': 'Impossible Foods',
        'Product': 'Impossible Spicy Sausage'
    },
    {
        'Calories': 230,
        'Protein': '13g',
        'Sodium': '630mg',
        'Saturated Fat': '7g',
        'Dietary Fiber': '5g',
        'Iron': '2.6mg',
        'Vitamin B12': '110%',
        'Cholesterol': '0mg',
        'URL': 'https://faq.impossiblefoods.com/hc/en-us/articles/5121845616151-What-are-the-nutrition-facts-for-Impossible-Sausage-Links-Meat-From-Plants',
        'Brand': 'Impossible Foods',
        'Product': 'Impossible Sausage Links Bratwurst'
    },
    {
        'Calories': 230,
        'Protein': '14g',
        'Sodium': '590mg',
        'Saturated Fat': '7g',
        'Dietary Fiber': '5g',
        'Iron': '2.7mg',
        'Vitamin B12': '110%',
        'Cholesterol': '0mg',
        'URL': 'https://faq.impossiblefoods.com/hc/en-us/articles/5121845616151-What-are-the-nutrition-facts-for-Impossible-Sausage-Links-Meat-From-Plants',
        'Brand': 'Impossible Foods',
        'Product': 'Impossible Sausage Links Italian'
    },
    {
        'Calories': 240,
        'Protein': '14g',
        'Sodium': '570mg',
        'Saturated Fat': '7g',
        'Dietary Fiber': '5g',
        'Iron': '2.8mg',
        'Vitamin B12': '110%',
        'Cholesterol': '0mg',
        'URL': 'https://faq.impossiblefoods.com/hc/en-us/articles/5121845616151-What-are-the-nutrition-facts-for-Impossible-Sausage-Links-Meat-From-Plants',
        'Brand': 'Impossible Foods',
        'Product': 'Impossible Sausage Links Spicy'
    },
    {
        'Calories': 180,
        'Protein': '10g',
        'Sodium': '350mg',
        'Saturated Fat': '1.5g',
        'Dietary Fiber': '2g',
        'Iron': '1.4mg',
        'Vitamin B12': '40%',
        'Cholesterol': '0mg',
        'URL': 'https://faq.impossiblefoods.com/hc/en-us/articles/8349646553111-What-are-the-nutrition-facts-for-Impossible-Chicken-Patties-Meat-From-Plants',
        'Brand': 'Impossible Foods',
        'Product': 'Impossible Chicken Patties'
    },
    {
        'Calories': 240,
        'Protein': '13g',
        'Sodium': '480mg',
        'Saturated Fat': '1.5g',
        'Dietary Fiber': '2g',
        'Iron': '2mg',
        'Vitamin B12': '25%',
        'Cholesterol': '0mg',
        'URL': 'https://faq.impossiblefoods.com/hc/en-us/articles/8349646553111-What-are-the-nutrition-facts-for-Impossible-Chicken-Patties-Meat-From-Plants',
        'Brand': 'Impossible Foods',
        'Product': 'Impossible Chicken Nuggets'
    },
    {
        'Calories': 180,
        'Protein': '11g',
        'Sodium': '430mg',
        'Saturated Fat': '1g',
        'Dietary Fiber': '4g',
        'Iron': '1.7mg',
        'Vitamin B12': '25%',
        'Cholesterol': '0mg',
        'URL': 'https://faq.impossiblefoods.com/hc/en-us/articles/12600526941719-What-are-the-nutrition-facts-for-Impossible-Chicken-Tenders-Meat-From-Plants',
        'Brand': 'Impossible Foods',
        'Product': 'Impossible Chicken Tenders'
    },
    {
        'Calories': 210,
        'Protein': '12g',
        'Sodium': '440mg',
        'Saturated Fat': '3.5g',
        'Dietary Fiber': '2g',
        'Iron': '2.2mg',
        'Vitamin B12': '70%',
        'Cholesterol': '0mg',
        'URL': 'https://faq.impossiblefoods.com/hc/en-us/articles/4413236407575-What-are-the-nutrition-facts-for-Impossible-Meatballs-Made-From-Plants',
        'Brand': 'Impossible Foods',
        'Product': 'Impossible Meatballs'
    }
]

In [35]:
# Morningstar

# we'll scrape from a third-party site, because the morningstar data is in images

# get urls to scrape
def getMorningstarUrls():
    response = requests.get("https://www.myfooddiary.com/brand/morningstar-farms")
    soup = BeautifulSoup(response.text, "html")

    # we'll append this to all the urls we find
    urlStart = "https://www.myfooddiary.com"

    # find all <a> tags such that href has morningstar-farms-
    links = soup.find_all("a", href=re.compile("morningstar-farms-"))

    # get the specific href parts
    all_urls = list(set([link["href"] for link in links]))

    # get full urls by appending urlStart to them
    full_urls = [urlStart + url for url in all_urls]

    return full_urls
    
# scrape the urls for nutrition information
def morningstarScraper(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html")

    # create a dictionary to store information about nutrients
    nutrientDict = {}

    # find the <a> tag with "nutrients/calories"
    calTag = soup.find("a", href=re.compile("/nutrients/calories"))

    # get caloric content
    nutrientDict["Calories"] = calTag.find_previous_sibling("div").text

    # now loop through the rest of the nutrients
    for nut in nutrients:
        # find a tag <a> such that inside is the exact string
        nut_label = soup.find("a", string=nut)

        try:
            # go to the next node, which is either text or a tag
            next_node = nut_label.next_sibling
            if hasattr(next_node, "text"):
                # if it has a text attribute, it's a tag, and we can get the text through .text
                nut_value = next_node.text.strip()
            else:
                # otherwise, it's just text
                nut_value = next_node.strip()

            # in case of whitespace as the next node
            if nut_value == "":
                span = nut_label.find_next_sibling("span")
                if span:
                    nut_value = span.text.strip()

            # add to dictionary
            nutrientDict[nut] = nut_value
        except:
            # if the nutrient isn't in there
            nutrientDict.update({nut:"N/A"})        

    # make sure we know what url this corresponds to
    nutrientDict["URL"] = url

    # add brand
    nutrientDict["Brand"] = "Morningstar"

    # fill in product 
    productName = "Morningstar " + url.split("morningstar-")[1].replace("-", " ").title()
    nutrientDict["Product"] = productName

    return nutrientDict



In [36]:
# get all the results

morningstarResults = []
for url in getMorningstarUrls():
    d = morningstarScraper(url)
    if d is not None:
        morningstarResults.append(d)
morningstarResults

[{'Calories': '120',
  'Protein': '9g',
  'Sodium': '220mg',
  'Saturated Fat': '0.5g',
  'Dietary Fiber': '4g',
  'Iron': '1.5mg',
  'Vitamin B12': 'N/A',
  'Cholesterol': '0mg',
  'URL': 'https://www.myfooddiary.com/foods/5140490/morningstar-farms-spicy-black-bean-burger',
  'Brand': 'Morningstar',
  'Product': 'Morningstar Farms Spicy Black Bean Burger'},
 {'Calories': '100',
  'Protein': '10g',
  'Sodium': '280mg',
  'Saturated Fat': '0g',
  'Dietary Fiber': '5g',
  'Iron': '1.4mg',
  'Vitamin B12': 'N/A',
  'Cholesterol': '0mg',
  'URL': 'https://www.myfooddiary.com/foods/7582001/morningstar-farms-garden-veggie-burger',
  'Brand': 'Morningstar',
  'Product': 'Morningstar Farms Garden Veggie Burger'},
 {'Calories': '190',
  'Protein': '12g',
  'Sodium': '460mg',
  'Saturated Fat': '1.5g',
  'Dietary Fiber': '2g',
  'Iron': '1.6mg',
  'Vitamin B12': 'N/A',
  'Cholesterol': '0mg',
  'URL': 'https://www.myfooddiary.com/foods/29512/morningstar-farms-chik-n-nuggets',
  'Brand': 'Morning

In [24]:
# Gardein

# we'll get all the URLs first, and then scrape them

# function that gets all pages with nutrition info
def getGardeinUrls():
    response = requests.get("https://www.gardein.com/sitemap.xml")
    soup = BeautifulSoup(response.text, "xml")

    # get all the URLs
    locs = soup.find_all("loc")
    all_urls = [loc.text for loc in locs]

    # get product URLs
    # five types of products
    product_urls = [url for url in all_urls 
                    if url.startswith("https://www.gardein.com/chickn-and-turky/") 
                    or url.startswith("https://www.gardein.com/beefless-and-porkless/")
                    or url.startswith("https://www.gardein.com/fishless/")
                    or url.startswith("https://www.gardein.com/meatless/")
                    or url.startswith("https://www.gardein.com/soups-chilis/")]

    return product_urls

# nutrients = ["Protein", "Sodium", "Saturated Fat", "Dietary Fiber",
          #  "Iron", "Vitamin B12", "Cholesterol"]

# <strong tabindex="0">Protein</strong> </span> <span class="col-xs-2 col-sm-2 nfp__values-right nfp__values-nowrap" tabindex="0">24 g</span>
# <div><span class="fontFix-topLevelFact header2" tabindex="0">850</span></div> <div class="sub-para fontFix-subparaTopLevel"><b tabindex="0">Sodium (mg)</b></div>
# <span class="col-xs-7 col-sm-4 nfp__content-indent-1" tabindex="0"> Saturated Fat </span> <span class="col-xs-2 col-sm-2 nfp__values-right nfp__values-nowrap" tabindex="0">1 g</span> 
# <span class="col-xs-7 col-sm-4 nfp__content-indent-1" tabindex="0"> Dietary Fiber </span> <span class="col-xs-2 col-sm-2 nfp__values-right nfp__values-nowrap" tabindex="0">1 g</span>
# <span class="col-xs-7 col-sm-4 customGetter" tabindex="0"> Iron </span> <span class="col-xs-2 col-sm-2 nfp__values-right nfp__values-nowrap" tabindex="0">4.1 mg</span>
# <strong tabindex="0">Cholesterol</strong> </span> <span class="col-xs-2 col-sm-2 nfp__values-right nfp__values-nowrap" tabindex="0">0 mg</span>


def gardeinScraper(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html")

    # find the smartlabel url
    # class is a reserved word in Python
    sl_tag = soup.find("a", class_="smartlabel-link")
    sl_url = sl_tag["data-smartlabel-info-url"]

    # scrape the sl url
    # use selenium because content is loaded via javascript
    options = Options()
    options.add_argument("--headless")
    driver = webdriver.Chrome(options=options)
    driver.get(sl_url)
    time.sleep(1)

    sl_soup = BeautifulSoup(driver.page_source, "html.parser")

    nutrientDict = {}

    # now loop through the rest of the nutrients
    for nut in nutrients:
        # find a tag <strong> such that inside is the exact string
        # deals with protein / cholesterol cases
        nut_label = sl_soup.find("strong", string=nut)
        if nut_label is not None:
            nut_value = nut_label.parent.find_next_sibling("span")
            nutrientDict.update({nut:nut_value.text})
        else:
            # find span tag such that inside is the string possibly w whitespace
            # deals with saturated fat, dietary fiber, iron
            nut_label = sl_soup.find("span", string=re.compile(nut))
            if nut_label is not None:
                nut_value = nut_label.find_next_sibling("span")
                nutrientDict.update({nut:nut_value.text})
            else:
                # find b tag such that inside is sodium
                nut_label = sl_soup.find("b", string=re.compile(nut))
                if nut_label is None:
                    nutrientDict.update({nut:"N/A"})
                else:
                    # get the parent of the <b> tag, then go back to the previous div, then find the <span> tag, then get the text inside
                    nut_value = nut_label.parent.find_previous_sibling("div").find("span")
                    nutrientDict.update({nut:nut_value.text})      
    
    # get caloric information
    cal_label = sl_soup.find("b", string=re.compile("Calories"))
    if cal_label is None:
        nutrientDict.update({"Calories":"N/A"})
    else:
        cal_value = cal_label.parent.find_previous_sibling("div").find("span")
        nutrientDict.update({"Calories":cal_value.text})

    # make sure we know what url this corresponds to
    nutrientDict["URL"] = url

    # add brand
    nutrientDict["Brand"] = "Gardein"

    # fill in product 
    productName = "Gardein " + url.split("/")[-1].replace("-"," ").title()
    nutrientDict["Product"] = productName

    return nutrientDict

    



In [ ]:
# get all the results

gardeinResults = []
for url in getGardeinUrls():
    d = gardeinScraper(url)
    if d is not None:
        gardeinResults.append(d)
gardeinResults

In [ ]:
# get rid of the NA one
del gardeinResults[-4]

In [38]:
# saving as CSV
all_results = beyondResults + impossibleResults + gardeinResults + morningstarResults
df = pd.DataFrame(all_results)
df.to_csv("nutrition_data.csv", index=False)
